In [78]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [79]:

mardan_path = "/content/drive/MyDrive/Colab Notebooks/Dr_Abdullah/Hb_Study_mardan.xlsx"
# KTH Dataset
kth_path = "/content/drive/MyDrive/Colab Notebooks/Dr_Abdullah/HB_STUDIE_kth.xlsx"

In [80]:
import pandas as pd

kth = pd.read_excel(kth_path)
mardan = pd.read_excel(mardan_path)

print(kth.shape)
print(mardan.shape)

(1261, 16384)
(426, 14)


In [81]:
print(kth.columns)
print(mardan.columns)

Index(['Age', 'Gender', 'Hb-A', 'HB-A2', 'Hb-F', 'Hb-S', 'Hb-D', 'RBC', 'HGB',
       'HCT',
       ...
       'Unnamed: 16374', 'Unnamed: 16375', 'Unnamed: 16376', 'Unnamed: 16377',
       'Unnamed: 16378', 'Unnamed: 16379', 'Unnamed: 16380', 'Unnamed: 16381',
       'Unnamed: 16382', 'Unnamed: 16383'],
      dtype='object', length=16384)
Index(['Age', 'Gender', 'Hb-A', 'HB-A2', 'Hb-F', 'Hb-S', 'Hb-D', 'RBC', 'HGB',
       'HCT', 'MCV', 'MCH', 'MCHC', 'Results'],
      dtype='object')


In [82]:
print("KTH")
print(kth["Results"].value_counts())

print("\n=========================\n")

print("Mardan")
print(mardan["Results"].value_counts())

KTH
Results
Normal                                 909
Beta thalassemia Minor                 193
Beta Thalsemia Minor                    91
Beta thalassemia major                  12
HbD Trait                                6
Borderline Hb A2                         5
Beta Thalsemia Major                     5
Borderline HbA2 level                    4
Sickle Beta Thalsemia                    4
HB D Trait                               4
Borderline HbA2 Level                    3
Sickle beta thalasemia                   3
sickle cell trait                        3
Sickle Cell Trait                        2
hb e 69.4                                1
Hb E trait                               1
Beta Thalsemia Minor                     1
hb e Beta Thalsemia                      1
Thalasemia                               1
Sickle cell beta thalasemia              1
Sickle Punjab                            1
Hb E Beta Heterozygous HB E 15.1%        1
sickle cell disease                      1

In [83]:
# Convert all diagnosis names to lowercase
kth["Results"] = kth["Results"].str.strip().str.lower()
mardan["Results"] = mardan["Results"].str.strip().str.lower()

In [84]:
kth["Results"] = kth["Results"].str.lower()
mardan["Results"] = mardan["Results"].str.lower()

In [85]:
# Standardized diagnosis mapping

diagnosis_mapping = {

    # -------------------------
    # Normal
    # -------------------------
    "normal": "Normal",

    # -------------------------
    # Beta Thalassemia Minor
    # -------------------------
    "beta thalassemia minor": "Beta Thalassemia Minor",
    "beta thalsemia minor": "Beta Thalassemia Minor",
    "beta talasemia minor": "Beta Thalassemia Minor",
    "beta thalaseimia minor": "Beta Thalassemia Minor",
    "beta thalassemiamior": "Beta Thalassemia Minor",
    "brta thaalasemia minor": "Beta Thalassemia Minor",
    "thalasemia": "Beta Thalassemia Minor",

    # -------------------------
    # Beta Thalassemia Major
    # -------------------------
    "beta thalassemia major": "Beta Thalassemia Major",
    "beta thalsemia major": "Beta Thalassemia Major",
    "thalasemia major": "Beta Thalassemia Major",

    # -------------------------
    # Borderline HbA2
    # -------------------------
    "borderline hba2": "Borderline HbA2",
    "borderline hb a2": "Borderline HbA2",
    "borderline hba2 level": "Borderline HbA2",
    "low hb a2": "Borderline HbA2",

    # -------------------------
    # Hb D Trait
    # -------------------------
    "hbd trait": "Hb D Trait",
    "hb d trait": "Hb D Trait",
    "hb d punjab trait": "Hb D Trait",

    # -------------------------
    # Sickle Disorders
    # -------------------------
    "sickle beta thalsemia": "Sickle Disorders",
    "sickle beta thalasemia": "Sickle Disorders",
    "sickle cell beta thalasemia": "Sickle Disorders",
    "heterozygous sickle beta thalasemia": "Sickle Disorders",
    "sickle cell trait": "Sickle Disorders",
    "sickle cell disease": "Sickle Disorders",
    "sickle punjab": "Sickle Disorders"

}

In [86]:
# Apply diagnosis mapping

kth["Results"] = kth["Results"].replace(diagnosis_mapping)
mardan["Results"] = mardan["Results"].replace(diagnosis_mapping)

In [87]:
mardan = mardan.drop_duplicates().reset_index(drop=True)

In [88]:
# ============================================================
# 5. Remove Rare Hb E Cases
# ============================================================

rare_classes = [
    "hb e trait",
    "hb e 69.4",
    "hb e beta thalsemia",
    "hb e beta heterozygous hb e 15.1%"
]

kth = kth[~kth["Results"].str.lower().isin(rare_classes)]
mardan = mardan[~mardan["Results"].str.lower().isin(rare_classes)]

In [89]:
# ============================================================
# 6. Add Hospital Source
# ============================================================

kth["Hospital"] = "KTH"
mardan["Hospital"] = "Mardan"

In [90]:
# ============================================================
# 7. Combine Datasets
# ============================================================

df = pd.concat([kth, mardan], ignore_index=True)

print("Combined Dataset Shape :", df.shape)

Combined Dataset Shape : (1608, 16385)


In [91]:
# ============================================================
# Remove Unnamed Columns
# ============================================================

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print(df.shape)

(1608, 15)


In [92]:
# ============================================================
# 8. Clean Gender
# ============================================================

df["Gender"] = (
    df["Gender"]
    .astype(str)
    .str.strip()
    .str.title()
)

df["Gender"] = df["Gender"].replace({
    "M": "Male",
    "F": "Female"
})

In [93]:
# ============================================================
# Convert Data Types
# ============================================================

numeric_columns = [
    "Age",
    "Hb-A",
    "HB-A2",
    "Hb-F",
    "Hb-S",
    "Hb-D",
    "RBC",
    "HGB",
    "HCT",
    "MCV",
    "MCH",
    "MCHC"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [94]:
print("Duplicates before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates after:", df.duplicated().sum())

Duplicates before: 69
Duplicates after: 0


In [95]:
# ============================================================
# Impute Missing Values Using Group-wise Median
# ============================================================

# Age
df["Age"] = df.groupby("Results")["Age"].transform(
    lambda x: x.fillna(x.median())
)

# Laboratory variables
lab_columns = [
    "Hb-A",
    "HB-A2",
    "Hb-D",
    "RBC",
    "HCT"
]

for col in lab_columns:
    df[col] = df.groupby("Results")[col].transform(
        lambda x: x.fillna(x.median())
    )

In [96]:
print("="*60)
print("Missing Values After Imputation")
print("="*60)

print(df.isnull().sum())

Missing Values After Imputation
Age         0
Gender      0
Hb-A        0
HB-A2       0
Hb-F        0
Hb-S        0
Hb-D        0
RBC         0
HGB         0
HCT         0
MCV         0
MCH         0
MCHC        0
Results     0
Hospital    0
dtype: int64


In [97]:
# Records with invalid gender values
invalid_gender = df[
    ~df["Gender"].isin(["Male", "Female"])
]

invalid_gender

,Age,Gender,Hb-A,HB-A2,Hb-F,Hb-S,Hb-D,RBC,HGB,HCT,MCV,MCH,MCHC,Results,Hospital
1121,13.0,97.8,2.2,3.98,0.0,0.0,0.0,5.00,33.50,84.1,27.4,22.0,32.6,Normal,KTH
1313,22.0,Nan,94.8,4.80,0.4,0.0,0.0,5.43,10.50,36.1,66.5,19.4,29.2,Beta Thalassemia Minor,Mardan
1318,22.0,Nan,97.0,5.90,1.1,0.0,0.0,5.89,11.80,38.4,65.1,20.0,30.7,Beta Thalassemia Minor,Mardan
1319,22.0,Nan,93.6,5.40,1.0,0.0,0.0,5.99,11.50,40.2,67.2,19.3,28.7,Beta Thalassemia Minor,Mardan
1320,22.0,Nan,94.7,5.20,0.1,0.0,0.0,5.08,9.85,32.9,64.7,19.4,29.9,Beta Thalassemia Minor,Mardan
1321,22.0,Nan,93.6,5.30,1.1,0.0,0.0,5.33,9.95,32.1,60.2,18.7,31.0,Beta Thalassemia Minor,Mardan
1340,20.0,Missing,95.9,2.20,1.9,0.0,0.0,3.64,10.60,26.9,73.9,29.1,39.3,Normal,Mardan
1342,20.0,Missing,97.6,2.10,0.3,0.0,0.0,4.81,7.70,29.4,61.1,16.0,26.2,Normal,Mardan
1385,20.0,Nan,97.7,2.20,0.1,0.0,0.0,4.60,7.80,26.4,57.3,17.1,29.7,Normal,Mardan
1388,20.0,Nan,95.9,2.20,1.9,0.0,0.0,3.64,10.60,26.9,73.9,29.1,39.3,Normal,Mardan


In [98]:
df["Gender"] = (
    df.groupby("Results")["Gender"]
      .transform(lambda x: x.fillna(x.mode().iloc[0]))
)

In [99]:
import numpy as np

# ============================================================
# Diagnosis-wise Gender Imputation
# ============================================================

df["Gender"] = df["Gender"].replace({
    "Nan": np.nan,
    "Missing": np.nan,
    "97.8": np.nan
})

df["Gender"] = (
    df.groupby("Results", observed=False)["Gender"]
      .transform(lambda x: x.fillna(x.mode().iloc[0]))
)

In [100]:
print(df["Gender"].value_counts(dropna=False))

Gender
Female    937
Male      602
Name: count, dtype: int64


In [101]:
print(df.isnull().sum())

Age         0
Gender      0
Hb-A        0
HB-A2       0
Hb-F        0
Hb-S        0
Hb-D        0
RBC         0
HGB         0
HCT         0
MCV         0
MCH         0
MCHC        0
Results     0
Hospital    0
dtype: int64


In [102]:
print(sorted(df["Results"].unique()))

['Beta Thalassemia Major', 'Beta Thalassemia Minor', 'Borderline HbA2', 'Hb D Trait', 'Normal', 'Sickle Disorders']


In [103]:
df["Gender"] = df["Gender"].astype("category")
df["Hospital"] = df["Hospital"].astype("category")
df["Results"] = df["Results"].astype("category")

In [104]:
print("="*60)
print("FINAL DATASET QUALITY CHECK")
print("="*60)

print("Shape:", df.shape)
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Records:", df.duplicated().sum())

print("\nData Types")
print(df.dtypes)

print("\nDiagnosis Distribution")
print(df["Results"].value_counts())

print("\nHospital Distribution")
print(df["Hospital"].value_counts())

print("\nGender Distribution")
print(df["Gender"].value_counts())

print("\nUnique Diagnoses")
print(sorted(df["Results"].unique()))

FINAL DATASET QUALITY CHECK
Shape: (1539, 15)
Missing Values: 0
Duplicate Records: 2

Data Types
Age          float64
Gender      category
Hb-A         float64
HB-A2        float64
Hb-F         float64
Hb-S         float64
Hb-D         float64
RBC          float64
HGB          float64
HCT          float64
MCV          float64
MCH          float64
MCHC         float64
Results     category
Hospital    category
dtype: object

Diagnosis Distribution
Results
Normal                    1118
Beta Thalassemia Minor     350
Beta Thalassemia Major      22
Borderline HbA2             22
Sickle Disorders            17
Hb D Trait                  10
Name: count, dtype: int64

Hospital Distribution
Hospital
KTH       1188
Mardan     351
Name: count, dtype: int64

Gender Distribution
Gender
Female    937
Male      602
Name: count, dtype: int64

Unique Diagnoses
['Beta Thalassemia Major', 'Beta Thalassemia Minor', 'Borderline HbA2', 'Hb D Trait', 'Normal', 'Sickle Disorders']


In [105]:
# Show duplicate rows
duplicates = df[df.duplicated(keep=False)]

duplicates.sort_values(by=["Results", "Hospital"])

,Age,Gender,Hb-A,HB-A2,Hb-F,Hb-S,Hb-D,RBC,HGB,HCT,MCV,MCH,MCHC,Results,Hospital
1340,20.0,Female,95.9,2.2,1.9,0.0,0.0,3.64,10.6,26.9,73.9,29.1,39.3,Normal,Mardan
1342,20.0,Female,97.6,2.1,0.3,0.0,0.0,4.81,7.7,29.4,61.1,16.0,26.2,Normal,Mardan
1388,20.0,Female,95.9,2.2,1.9,0.0,0.0,3.64,10.6,26.9,73.9,29.1,39.3,Normal,Mardan
1389,20.0,Female,97.6,2.1,0.3,0.0,0.0,4.81,7.7,29.4,61.1,16.0,26.2,Normal,Mardan


In [106]:
dup = df[df.duplicated(keep=False)]

print(dup.groupby(list(df.columns)).size())

Age   Gender  Hb-A  HB-A2  Hb-F  Hb-S  Hb-D  RBC   HGB   HCT   MCV   MCH   MCHC  Results                 Hospital
20.0  Female  95.9  2.1    0.3   0.0   0.0   3.64  7.7   26.9  61.1  16.0  26.2  Beta Thalassemia Major  KTH         0
                                                                                                         Mardan      0
                                                                                 Beta Thalassemia Minor  KTH         0
                                                                                                         Mardan      0
                                                                                 Borderline HbA2         KTH         0
                                                                                                                    ..
      Male    97.6  2.2    1.9   0.0   0.0   4.81  10.6  29.4  73.9  29.1  39.3  Hb D Trait              Mardan      0
                                                     

/tmp/ipykernel_1970/75816618.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(dup.groupby(list(df.columns)).size())


In [107]:
# Show only the duplicated rows
duplicates = df[df.duplicated(keep=False)]

duplicates.sort_values(by=["Age", "Hb-A", "Hospital"])

,Age,Gender,Hb-A,HB-A2,Hb-F,Hb-S,Hb-D,RBC,HGB,HCT,MCV,MCH,MCHC,Results,Hospital
1340,20.0,Female,95.9,2.2,1.9,0.0,0.0,3.64,10.6,26.9,73.9,29.1,39.3,Normal,Mardan
1388,20.0,Female,95.9,2.2,1.9,0.0,0.0,3.64,10.6,26.9,73.9,29.1,39.3,Normal,Mardan
1342,20.0,Female,97.6,2.1,0.3,0.0,0.0,4.81,7.7,29.4,61.1,16.0,26.2,Normal,Mardan
1389,20.0,Female,97.6,2.1,0.3,0.0,0.0,4.81,7.7,29.4,61.1,16.0,26.2,Normal,Mardan


In [108]:
print(mardan.shape)

print(mardan.duplicated().sum())

(351, 15)
0


In [109]:
mardan[mardan.duplicated(keep=False)]

,Age,Gender,Hb-A,HB-A2,Hb-F,Hb-S,Hb-D,RBC,HGB,HCT,MCV,MCH,MCHC,Results,Hospital


In [110]:
# Number of occurrences of each unique row
dup_summary = (
    mardan.groupby(mardan.columns.tolist(), dropna=False)
           .size()
           .reset_index(name="Count")
)

dup_summary = dup_summary[dup_summary["Count"] > 1]

print(dup_summary["Count"].value_counts().sort_index())
dup_summary.head(20)

Series([], Name: count, dtype: int64)


,Age,Gender,Hb-A,HB-A2,Hb-F,Hb-S,Hb-D,RBC,HGB,HCT,MCV,MCH,MCHC,Results,Hospital,Count


In [112]:
# Convert categorical columns
categorical_cols = ["Gender", "Hospital", "Results"]

for col in categorical_cols:
    df[col] = df[col].astype("category")

print(df.dtypes)

Age          float64
Gender      category
Hb-A         float64
HB-A2        float64
Hb-F         float64
Hb-S         float64
Hb-D         float64
RBC          float64
HGB          float64
HCT          float64
MCV          float64
MCH          float64
MCHC         float64
Results     category
Hospital    category
dtype: object


04_Model_Explainability_SHAP.ipynb
05_Statistical_Validation.ipynb

In [114]:
# ============================================================
# Save Final Preprocessed Dataset
# ============================================================

save_path_excel = "/content/drive/MyDrive/Colab Notebooks/Dr_Abdullah/Hemoglobinopathy_Final_Preprocessed.xlsx"
save_path_csv = "/content/drive/MyDrive/Colab Notebooks/Dr_Abdullah/Hemoglobinopathy_Final_Preprocessed.csv"

# Save
df.to_excel(save_path_excel, index=False)
df.to_csv(save_path_csv, index=False)

print("Dataset saved successfully!")

Dataset saved successfully!
